In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "6"

import torch

from tts.config.ndaligner.training_module_config import NDAlignerTrainingModuleConfigs
from tts.config.utils.io import load_config
from tts.models.ndaligner import init_nd_aligner_training_module

/home/blue2959/monotonic_tts/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## INIT Models

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
aligner_training_module_cfg_path = "./runs/nd_aligner_vctk_coupling_lambda_search_20260724-002812/model_config.json"
aligner_training_module_ckpt_path = "./runs/nd_aligner_vctk_coupling_lambda_search_20260724-002812/checkpoints_timit_bae/best_step_timit_bae_0.020375_step_141000_epoch_39.pth"


model_config = load_config(aligner_training_module_cfg_path, NDAlignerTrainingModuleConfigs,)

aligner_training_module = init_nd_aligner_training_module(
    config=model_config,
    load_vocoder=True,
    device=device,
)
aligner_training_module.load_checkpoint(
    ckpt_path=aligner_training_module_ckpt_path,
    device=device,
)
aligner = aligner_training_module.nd_aligner.eval()

/home/blue2959/monotonic_tts/.venv/lib/python3.13/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loading nested state_dict from key 'model' in ./runs/nd_aligner_vctk_coupling_lambda_search_20260724-002812/checkpoints_timit_bae/best_step_timit_bae_0.020375_step_141000_epoch_39.pth
✅ All weights matched perfectly.
Checkpoint loading process finished.


## INIT BenchMarkers (TIMIT)

In [3]:
# from silero_vad import load_silero_vad

# assert aligner.input_maker is not None

# aligner.input_maker.silero_model = load_silero_vad(onnx=True)
# aligner.input_maker.trim_nonspeech_region = aligner.input_maker.zero_nonspeech_region = True

In [4]:
from tts.benchmark.timit.benchmarker import TIMITBenchMarker

TIMIT_ROOT = "/shared/data_zfs/blue2959/TIMIT/TEST"
assert aligner.input_maker is not None

timit_benchmarker = TIMITBenchMarker(
    root_dir=TIMIT_ROOT,
    ref_audio_sr=16_000,
    hyp_audio_sr=22_050,
    hyp_hop_length=256,
    input_maker=aligner.input_maker,
    hyp_ignore_symbols=aligner.input_maker.tokenizer.ignore_symbols,
    max_ref_words_per_hyp_word=5,
)

[TIMITBenchMarker] Found 1680 valid (WRD, WAV, TXT) triplets.


In [ ]:
# w_st = 1.0

with torch.no_grad():
    metrics = timit_benchmarker(
        aligner=aligner,
        vocoder=aligner_training_module.vocoder,
        max_test_samples=None,
    )

print(f"{metrics.word_boundary_error * 1000:.2f} ms")
print(f"{metrics.p_word_100ms:.2f} %")
print(f"{metrics.p_word_50ms:.2f} %")
print(f"{metrics.p_word_25ms:.2f} %")
print(f"{metrics.p_word_10ms:.2f} %")

Computing Alignments: 100%|██████████| 1680/1680 [00:44<00:00, 37.96it/s]

20.89 ms
98.65 %
91.74 %
73.35 %
37.35 %


In [ ]:
# w_st = 0.75

device = 'cuda' if torch.cuda.is_available() else 'cpu'
aligner_training_module_cfg_path = "/home/blue2959/monotonic_tts/runs/nd_aligner_vctk_coupling_lambda_search_20260723-125106/model_config.json"
aligner_training_module_ckpt_path = "/home/blue2959/monotonic_tts/runs/nd_aligner_vctk_coupling_lambda_search_20260723-125106/checkpoints_timit_bae/best_step_timit_bae_0.020652_step_101500_epoch_28.pth"


model_config = load_config(aligner_training_module_cfg_path, NDAlignerTrainingModuleConfigs,)

aligner_training_module = init_nd_aligner_training_module(
    config=model_config,
    load_vocoder=True,
    device=device,
)
aligner_training_module.load_checkpoint(
    ckpt_path=aligner_training_module_ckpt_path,
    device=device,
)
aligner = aligner_training_module.nd_aligner.eval()

/home/blue2959/monotonic_tts/.venv/lib/python3.13/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loading nested state_dict from key 'model' in /home/blue2959/monotonic_tts/runs/nd_aligner_vctk_coupling_lambda_search_20260723-125106/checkpoints_timit_bae/best_step_timit_bae_0.020652_step_101500_epoch_28.pth
✅ All weights matched perfectly.
Checkpoint loading process finished.


In [7]:
with torch.no_grad():
    metrics = timit_benchmarker(
        aligner=aligner,
        vocoder=aligner_training_module.vocoder,
        max_test_samples=None,
    )

print(f"{metrics.word_boundary_error * 1000:.2f} ms")
print(f"{metrics.p_word_100ms:.2f} %")
print(f"{metrics.p_word_50ms:.2f} %")
print(f"{metrics.p_word_25ms:.2f} %")
print(f"{metrics.p_word_10ms:.2f} %")

Computing Alignments: 100%|██████████| 1680/1680 [00:37<00:00, 44.29it/s]

20.86 ms
98.31 %
91.79 %
73.79 %
37.82 %


## INIT BenchMarkers (Buckeye)

In [5]:
from tts.benchmark.timit.benchmarker import TIMITBenchMarker
assert aligner.input_maker is not None


BUCKEYE_ROOT = "/shared/data_zfs/blue2959/Buckeye-grid" # (compatible with timit benchmarker!)

buckeye_benchmarker = TIMITBenchMarker(
    root_dir=BUCKEYE_ROOT,
    ref_audio_sr=16_000,
    hyp_audio_sr=22_050,
    hyp_hop_length=256,
    input_maker=aligner.input_maker,
    hyp_ignore_symbols=aligner.input_maker.tokenizer.ignore_symbols,
    max_ref_words_per_hyp_word=5,
)

[TIMITBenchMarker] Found 19273 valid (WRD, WAV, TXT) triplets.


In [7]:
with torch.inference_mode():
    metrics = buckeye_benchmarker(
        aligner=aligner,
        vocoder=None,
        max_test_samples=1000,
    )

print(f"{metrics.word_boundary_error * 1000:.2f} ms")
print(f"{metrics.p_word_100ms:.2f} %")
print(f"{metrics.p_word_50ms:.2f} %")
print(f"{metrics.p_word_25ms:.2f} %")
print(f"{metrics.p_word_10ms:.2f} %")

Computing Alignments: 100%|██████████| 1000/1000 [00:30<00:00, 32.84it/s]

25.73 ms
95.05 %
89.76 %
74.08 %
39.99 %
